# Pacific Labeled Corals EDA

Exploratory analysis of the S3-hosted Pacific Labeled Corals dataset produced by
`verify_raw_data_and_add_to_s3.py`.

All inputs are read from S3 (`s3://dev-datamermaid-sm-sources/external_validation_datasets/pacific_labeled_corals/`) — no local files needed.

The Moorea split of the original Pacific Labeled Corals release is processed separately under `mermaidseg/datasets/moorea_labeled_corals`; only `heron_reef`, `line_islands` and `nanwan_bay` are present here.

In [ ]:
import json

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

from mermaidseg.datasets.pacific_labeled_corals import PacificLabeledCoralsDataset

In [ ]:
BUCKET = "dev-datamermaid-sm-sources"
PREFIX = "external_validation_datasets/pacific_labeled_corals"

s3 = boto3.client("s3")


def s3_get_json(key: str) -> dict:
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    return json.loads(body)


classes = s3_get_json(f"{PREFIX}/classes.json")
colors = s3_get_json(f"{PREFIX}/colors.json")
manifest = s3_get_json(f"{PREFIX}/manifest.json")

print(f"Num classes (incl. unlabeled): {len(classes)}")
print(f"Manifest dataset: {manifest['dataset_name']} created at {manifest['created_at_utc']}")
print(
    f"Annotations parquet: s3://{manifest['s3']['bucket']}/{manifest['s3']['annotations_parquet']}"
)

## Dataset summary (from annotations parquet)

Per-site counts are derived from the parquet on S3.

In [ ]:
ANNOTATOR_COLS = ["archived", "host", "visitor1", "visitor2", "visitor3", "visitor4", "visitor5"]

parquet_uri = f"s3://{manifest['s3']['bucket']}/{manifest['s3']['annotations_parquet']}"
df_meta = pd.read_parquet(parquet_uri)


def _label_classes_present(group: pd.DataFrame) -> int:
    cols = [c for c in ANNOTATOR_COLS if c in group.columns]
    return int(group[cols].melt().nunique()) if cols else 0


sites_df = (
    df_meta.groupby("site", sort=True)
    .apply(
        lambda g: pd.Series(
            {
                "num_images": g.drop_duplicates(["subset", "image_id"]).shape[0],
                "num_annotations": len(g),
                "num_label_classes_present": _label_classes_present(g),
            }
        ),
        include_groups=False,
    )
    .rename_axis("site")
)

sites_df.loc["TOTAL"] = {
    "num_images": df_meta.drop_duplicates(["site", "subset", "image_id"]).shape[0],
    "num_annotations": len(df_meta),
    "num_label_classes_present": _label_classes_present(df_meta),
}
sites_df

## Instantiate the dataset

`annotator_column="host"` is the default; reference subsets (which only carry `archived`) are kept thanks to the default `fallback_annotator="archived"`. The actual annotator chosen per row is recorded in `df_annotations["annotator_used"]`.

In [ ]:
dataset = PacificLabeledCoralsDataset(
    source_bucket=BUCKET,
    source_s3_prefix=PREFIX,
    annotations_path=manifest["s3"]["annotations_parquet"],
    padding=7,
)
print(f"Dataset length: {len(dataset)}")
print(f"Num source classes (incl. background): {dataset.num_source_classes}")
print(
    f"annotator_column={dataset.annotator_column!r} | "
    f"fallback_annotator={dataset.fallback_annotator!r}"
)
print(
    "annotator_used counts:\n",
    dataset.df_annotations["annotator_used"].value_counts().to_string(),
)
dataset.df_annotations.head()

## Sample image + sparse annotations per site

In [ ]:
sites = sorted(dataset.df_images["site"].unique())
n = len(sites)
ncols = min(3, n) if n > 0 else 1
nrows = int(np.ceil(n / ncols)) if n > 0 else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)

rng = np.random.default_rng(0)
for ax, site in zip(axes.ravel(), sites, strict=False):
    site_rows = dataset.df_images.index[dataset.df_images["site"] == site].to_numpy()
    if len(site_rows) == 0:
        ax.axis("off")
        ax.set_title(f"{site}: no images")
        continue
    idx = int(rng.choice(site_rows))
    image_id = dataset.df_images.loc[idx, "image_id"]
    subset = dataset.df_images.loc[idx, "subset"]
    image_ext = dataset.df_images.loc[idx, "image_ext"]
    image = dataset.read_image(image_id=image_id, site=site, subset=subset, image_ext=image_ext)

    ann_mask = (
        (dataset.df_annotations["site"] == site)
        & (dataset.df_annotations["subset"] == subset)
        & (dataset.df_annotations["image_id"] == image_id)
    )
    anns = dataset.df_annotations.loc[ann_mask, ["row", "col", "source_label_name"]]

    ax.imshow(image)
    seen_labels = []
    for _, r in anns.iterrows():
        rgb = colors.get(r["source_label_name"], [255, 0, 0])
        ax.scatter(
            r["col"],
            r["row"],
            color=np.array(rgb) / 255.0,
            s=30,
            edgecolors="white",
            linewidths=0.4,
        )
        seen_labels.append(r["source_label_name"])
    handles = [
        Patch(facecolor=np.array(colors.get(name, [255, 0, 0])) / 255.0, label=name)
        for name in sorted(set(seen_labels))[:10]
    ]
    ax.set_title(f"{site}/{subset} / {image_id} ({len(anns)} pts)")
    ax.axis("off")
    if handles:
        ax.legend(handles=handles, loc="upper right", fontsize=6, framealpha=0.7)

for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Top-20 label classes per site

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
for ax, site in zip(axes.ravel(), sites, strict=False):
    counts = (
        dataset.df_annotations.loc[dataset.df_annotations["site"] == site, "source_label_name"]
        .value_counts()
        .head(20)
    )
    if counts.empty:
        ax.set_title(f"{site}: no annotations")
        ax.axis("off")
        continue
    bar_colors = [np.array(colors.get(name, [128, 128, 128])) / 255.0 for name in counts.index]
    ax.barh(range(len(counts)), counts.to_numpy()[::-1], color=bar_colors[::-1])
    ax.set_yticks(range(len(counts)))
    ax.set_yticklabels(counts.index[::-1], fontsize=8)
    site_mask = dataset.df_annotations["site"] == site
    n_unique = dataset.df_annotations.loc[site_mask, "source_label_name"].nunique()
    ax.set_title(f"{site} (top 20 of {n_unique})")
for ax in axes.ravel()[len(sites) :]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Sanity-check `__getitem__`

Confirm that `dataset[idx]` returns a `(image, source_labels)` pair where the source-labels mask is rendered consistently with `BaseCoralDataset` semantics.

In [ ]:
idx = 0
image, source_mask = dataset[idx]
row = dataset.df_images.loc[idx]
print(
    f"Image idx={idx}: site={row['site']}, subset={row['subset']}, "
    f"image_id={row['image_id']}{row['image_ext']}"
)
print(f"Image shape: {image.shape}, mask shape: {source_mask.shape}")
print(
    f"Unique mask ids (excl. 0): {sorted({int(v) for v in np.unique(source_mask) if v != 0})[:20]}"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image)
axes[0].set_title("Image")
axes[0].axis("off")

vis = np.zeros((*source_mask.shape, 3), dtype=np.uint8)
for local_id, name in dataset.source_id2name.items():
    rgb = colors.get(name, [255, 0, 0])
    vis[source_mask == local_id] = np.array(rgb, dtype=np.uint8)

axes[1].imshow(image)
axes[1].imshow(vis, alpha=0.6)
axes[1].set_title("Source-label mask overlay")
axes[1].axis("off")
plt.tight_layout()
plt.show()